# MS MARCO and TREC-CAR evaluation

In [1]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch

print(torch.cuda.device_count())     
print(torch.cuda.current_device())    
print(torch.cuda.get_device_name(0)) 

import numpy as np
from sklearn.metrics import ndcg_score
from colbert import Indexer, Searcher
from colbert.data import Queries
from colbert.infra import ColBERTConfig, Run, RunConfig


MSMARCO_DIR = "/ml_data/aryan/msmarco_sample"
TRECCAR_DIR = "/ml_data/aryan/trec-car_sample"

1
0
NVIDIA H200 NVL


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
import transformers
transformers.__version__

'4.48.3'

In [3]:
def compute_metrics(hits: dict, rel: dict):
    ndcg_scores = []
    rr = []
    recalls = []
    ap_scores = []
    for qid, ret_doc in hits.items():
        rel_doc = rel[qid]

        y_true = np.array([[1 if doc in rel_doc else 0 for doc in ret_doc[:10]]])
        y_score = np.array([[len(ret_doc) - i for i in range(len(ret_doc[:10]))]])
        ndcg = ndcg_score(y_true, y_score, k=10)
        ndcg_scores.append(ndcg)

        top100 = set(ret_doc)
        count = 0
        for doc in rel_doc:
            if doc in top100:
                count += 1
        recall = count / len(rel_doc) if rel_doc else 0
        recalls.append(recall)

        rank = 0
        for i, doc in enumerate(ret_doc[:10]):
            if doc in rel_doc:
                rank = i + 1
                break
        rr.append(1 / rank if rank > 0 else 0)

        num_hits = 0
        precisions = []
        for i, doc in enumerate(ret_doc[:10]):
            if doc in rel_doc:
                num_hits += 1
                precisions.append(num_hits / (i + 1))
        ap = sum(precisions) / min(len(rel_doc), 10) if rel_doc else 0
        ap_scores.append(ap)

    avg_ndcg = np.mean(np.array(ndcg_scores))
    print(f'NDCG@10: {avg_ndcg:.4f}')

    mrr = np.mean(np.array(rr))
    print(f'MRR@10: {mrr:.4f}')

    recall = np.mean(np.array(recalls))
    print(f'Recall@100: {recall:.4f}')

    avg_ap = np.mean(np.array(ap_scores))
    print(f'MAP@10: {avg_ap:.4f}')

    return avg_ndcg, mrr, recall, avg_ap

In [4]:
dataset_name = 'msmarco_sample'
data_dir = Path(MSMARCO_DIR)
outpath = data_dir
i_to_id = {}
os.makedirs(outpath, exist_ok=True)
with open(data_dir / 'sampled_collection.tsv', encoding='utf-8') as source, open(outpath / 'collections.tsv', 'w', encoding='utf-8') as target:
    for i, line in enumerate(source):
        id_, text = line.rstrip('\r\n').split('\t', maxsplit=1)
        i_to_id[i] = id_
        target.write(f'{i}\t{text}\n')

In [5]:
with open(outpath / 'i-to-ids.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(i_to_id, ensure_ascii=False))

In [6]:
i_to_qid = {}
with open(data_dir / 'sampled_queries.dev.tsv', encoding='utf-8') as source, open(outpath / 'queries_test.tsv', 'w', encoding='utf-8') as target:
    for i, line in enumerate(source):
        qid_, query = line.rstrip('\r\n').split('\t', maxsplit=1)
        i_to_qid[i] = qid_
        target.write(f'{i}\t{query}\n')

In [7]:
with open(outpath / 'i-to-qids.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(i_to_qid, ensure_ascii=False))

In [8]:
def main():
    with Run().context(RunConfig(nranks=1, experiment='msmarco-sample')):
        config = ColBERTConfig(
            nbits=2,
            root='/ml_data/aryan',
        )
        indexer = Indexer(checkpoint='jinaai/jina-colbert-v1-en', config=config)
        indexer.index(name='msmarco-sample.nbits=2', collection=str(outpath / 'collections.tsv'), overwrite='force_silent_overwrite')


if __name__ == '__main__':
    main()



[Oct 02, 13:32:45] #> Creating directory /home/aryan/ColBERT/evaluation/experiments/msmarco-sample/indexes/msmarco-sample.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 1e-5,
    "maxsteps": 400000,
    "save_every": null,
    "warmup": 20000,
    "warmup_bert": null,
    "relu": false,
    "nway": 64,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/ji

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



[Oct 02, 13:33:00] [0] 		 # of sampled PIDs = 175272 	 sampled_pids[:3] = [436857, 768276, 10663]
[Oct 02, 13:33:00] [0] 		 #> Encoding 175272 passages..


...

[Oct 02, 14:01:17] [0] 		 #> Checking all files were saved...
[Oct 02, 14:01:17] [0] 		 Found all files!
[Oct 02, 14:01:17] [0] 		 #> Building IVF...
[Oct 02, 14:01:17] [0] 		 #> Loading codes...


100%|██████████| 40/40 [00:00<00:00, 228.75it/s]


[Oct 02, 14:01:17] [0] 		 Sorting codes...
[Oct 02, 14:01:18] [0] 		 Getting unique codes...
[Oct 02, 14:01:18] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 02, 14:01:18] #> Building the emb2pid mapping..
[Oct 02, 14:01:20] len(emb2pid) = 66821756


100%|██████████| 65536/65536 [00:01<00:00, 43124.52it/s]


[Oct 02, 14:01:22] #> Saved optimized IVF to /home/aryan/ColBERT/evaluation/experiments/msmarco-sample/indexes/msmarco-sample.nbits=2/ivf.pid.pt
[Oct 02, 14:01:22] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/evaluation/experiments/msmarco-sample/indexes/msmarco-sample.nbits=2/metadata.json ..

#> Joined...


In [9]:
i_to_qid = {}
with open(data_dir / 'sampled_queries.dev.tsv', encoding='utf-8') as source, open(outpath / 'queries_test.tsv', 'w', encoding='utf-8') as target:
    for i, line in enumerate(source):
        qid_, query = line.rstrip('\r\n').split('\t', maxsplit=1)
        i_to_qid[i] = qid_
        target.write(f'{i}\t{query}\n')

with open(outpath / 'i-to-qids.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(i_to_qid, ensure_ascii=False))

In [10]:
with Run().context(RunConfig(nranks=1, experiment='msmarco-sample')):
    config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(index='msmarco-sample.nbits=2', config=config, checkpoint='jinaai/jina-colbert-v1-en')
    queries = Queries(str(outpath / 'queries_test.tsv'))
    ranking = searcher.search_all(queries, k=100)
    ranking.save('msmarco-sample.nbits=2.ranking.tsv')

[Oct 02, 14:01:45] #> Loading collection...
0M 
[Oct 02, 14:02:02] #> Loading codec...
[Oct 02, 14:02:02] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


[Oct 02, 14:02:02] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 02, 14:02:02] #> Loading IVF...
[Oct 02, 14:02:02] #> Loading doclens...


100%|██████████████████| 40/40 [00:00<00:00, 2934.82it/s]

[Oct 02, 14:02:02] #> Loading codes and residuals...



100%|████████████████████| 40/40 [00:01<00:00, 22.50it/s]

[Oct 02, 14:02:04] #> Loading the queries from /ml_data/aryan/msmarco_sample/queries_test.tsv ...


[Oct 02, 14:02:04] #> Got 6980 queries. All QIDs are unique.



/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
6980it [27:45,  4.19it/s]




[Oct 02, 14:29:53] #> Creating directory /home/aryan/ColBERT/evaluation/experiments/msmarco-sample/none/2026-10/02/13.32.28 


[Oct 02, 14:29:54] #> Saved ranking of 6980 queries and 698000 lines to /home/aryan/ColBERT/evaluation/experiments/msmarco-sample/none/2026-10/02/13.32.28/msmarco-sample.nbits=2.ranking.tsv


In [11]:
qrels = defaultdict(list)
with open(data_dir / 'sampled_qrels.dev.tsv', encoding='utf-8') as f:
    for line in f:
        qid_, _, docid_, rel_ = line.rstrip('\r\n').split('\t')
        if int(rel_) > 0:
            qrels[qid_].append(docid_)

In [18]:
preds = defaultdict(list)
for qid, results in ranking.items():
    actual_qid = i_to_qid[qid]
    for docid, _, _ in results[:50]:
        preds[actual_qid].append(i_to_id[docid])

compute_metrics(preds, qrels);

NDCG@10: 0.7556
MRR@10: 0.7165
Recall@100: 0.9442
MAP@10: 0.7095


# TREC-CAR evaluation

In [19]:
dataset_name = 'trec-car_sample'
data_dir = Path(TRECCAR_DIR)
outpath = data_dir
i_to_id = {}
os.makedirs(outpath, exist_ok=True)
with open(data_dir / 'sampled_collection.tsv', encoding='utf-8') as source, open(outpath / 'collections.tsv', 'w', encoding='utf-8') as target:
    for i, line in enumerate(source):
        id_, text = line.rstrip('\r\n').split('\t', maxsplit=1)
        i_to_id[i] = id_
        target.write(f'{i}\t{text}\n')

with open(outpath / 'i-to-ids.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(i_to_id, ensure_ascii=False))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [20]:
def main():
    with Run().context(RunConfig(nranks=1, experiment='trec-car-sample')):
        config = ColBERTConfig(
            nbits=2,
            root='/ml_data/aryan',
        )
        indexer = Indexer(checkpoint='jinaai/jina-colbert-v1-en', config=config)
        indexer.index(name='trec-car-sample.nbits=2', collection=str(outpath / 'collections.tsv'), overwrite='force_silent_overwrite')


if __name__ == '__main__':
    main()



[Oct 02, 14:37:12] #> Creating directory /home/aryan/ColBERT/evaluation/experiments/trec-car-sample/indexes/trec-car-sample.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 1e-5,
    "maxsteps": 400000,
    "save_every": null,
    "warmup": 20000,
    "warmup_bert": null,
    "relu": false,
    "nway": 64,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/ji

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



[Oct 02, 14:37:27] [0] 		 # of sampled PIDs = 175272 	 sampled_pids[:3] = [436857, 768276, 10663]
[Oct 02, 14:37:27] [0] 		 #> Encoding 175272 passages..


...

[Oct 02, 15:03:07] [0] 		 #> Checking all files were saved...
[Oct 02, 15:03:07] [0] 		 Found all files!
[Oct 02, 15:03:07] [0] 		 #> Building IVF...
[Oct 02, 15:03:07] [0] 		 #> Loading codes...


100%|██████████| 40/40 [00:00<00:00, 234.93it/s]


[Oct 02, 15:03:07] [0] 		 Sorting codes...
[Oct 02, 15:03:08] [0] 		 Getting unique codes...
[Oct 02, 15:03:08] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 02, 15:03:08] #> Building the emb2pid mapping..
[Oct 02, 15:03:10] len(emb2pid) = 66675299


100%|██████████| 65536/65536 [00:01<00:00, 44049.19it/s]


[Oct 02, 15:03:11] #> Saved optimized IVF to /home/aryan/ColBERT/evaluation/experiments/trec-car-sample/indexes/trec-car-sample.nbits=2/ivf.pid.pt
[Oct 02, 15:03:12] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/evaluation/experiments/trec-car-sample/indexes/trec-car-sample.nbits=2/metadata.json ..

#> Joined...


In [21]:
i_to_qid = {}
with open(data_dir / 'sampled_queries.test.tsv', encoding='utf-8') as source, open(outpath / 'queries_test.tsv', 'w', encoding='utf-8') as target:
    for i, line in enumerate(source):
        qid_, query = line.rstrip('\r\n').split('\t', maxsplit=1)
        i_to_qid[i] = qid_
        target.write(f'{i}\t{query}\n')

with open(outpath / 'i-to-qids.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(i_to_qid, ensure_ascii=False))



In [22]:
with Run().context(RunConfig(nranks=1, experiment='trec-car-sample')):
    config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(index='trec-car-sample.nbits=2', config=config, checkpoint='jinaai/jina-colbert-v1-en')
    queries = Queries(str(outpath / 'queries_test.tsv'))
    ranking = searcher.search_all(queries, k=50)
    ranking.save('trec-car-sample.nbits=2.ranking.tsv')

[Oct 02, 15:03:13] #> Loading collection...
0M 
[Oct 02, 15:03:31] #> Loading codec...
[Oct 02, 15:03:32] #> Loading IVF...
[Oct 02, 15:03:32] #> Loading doclens...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|██████████████████| 40/40 [00:00<00:00, 2667.28it/s]


[Oct 02, 15:03:32] #> Loading codes and residuals...


100%|████████████████████| 40/40 [00:01<00:00, 21.87it/s]


[Oct 02, 15:03:34] #> Loading the queries from /ml_data/aryan/trec-car_sample/queries_test.tsv ...
[Oct 02, 15:03:34] #> Got 2125 queries. All QIDs are unique.



/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
2125it [02:13, 15.93it/s] 




[Oct 02, 15:05:49] #> Creating directory /home/aryan/ColBERT/evaluation/experiments/trec-car-sample/none/2026-10/02/13.32.28 


[Oct 02, 15:05:49] #> Saved ranking of 2125 queries and 106250 lines to /home/aryan/ColBERT/evaluation/experiments/trec-car-sample/none/2026-10/02/13.32.28/trec-car-sample.nbits=2.ranking.tsv


In [24]:
qrels = defaultdict(list)
with open(data_dir / 'sampled_qrels.test.tsv', encoding='utf-8') as f:
    for line in f:
        qid_, _, docid_, rel_ = line.rstrip('\r\n').split('\t')
        if int(rel_) > 0:
            qrels[qid_].append(docid_)



In [26]:
preds = defaultdict(list)
for qid, results in ranking.items():
    actual_qid = i_to_qid[qid]
    for docid, _, _ in results:
        preds[actual_qid].append(i_to_id[docid])

compute_metrics(preds, qrels);

NDCG@10: 0.6176
MRR@10: 0.5735
Recall@100: 0.7825
MAP@10: 0.4124
